In [0]:
# %sql
# this table is  SCD Type 1 (with Audit Columns)
# create or replace table intellibi_catalog.intellibi_gold.MonthlySales (
# sale_id integer,
# product string,
# category string,
# quantity integer,
# price double,
# sale_date date,
# region string,
# initial_load_ts timestamp,
# last_update_ts timestamp
# );

In [0]:
%sql
MERGE INTO intellibi_catalog.intellibi_gold.MonthlySales AS target
USING (
  SELECT *
  FROM intellibi_catalog.intellibi_silver.cleanedMonthlySales
  QUALIFY ROW_NUMBER() OVER (PARTITION BY sale_id ORDER BY ingest_ts DESC) = 1
) AS source
ON target.sale_id = source.sale_id

-- 1. Update existing records: keep initial_load_ts untouched, update last_update_ts from Silver's ingest_ts
WHEN MATCHED AND (
    NOT (target.product   <=> source.product)   OR
    NOT (target.category  <=> source.category)  OR
    NOT (target.quantity  <=> source.quantity)  OR
    NOT (target.price     <=> source.price)     OR
    NOT (target.sale_date <=> source.sale_date) OR
    NOT (target.region    <=> source.region)
) THEN
  UPDATE SET
    target.product        = source.product,
    target.category       = source.category,
    target.quantity       = source.quantity,
    target.price          = source.price,
    target.sale_date      = source.sale_date,
    target.region         = source.region,
    target.last_update_ts = source.ingest_ts

-- 2. Insert new records: map Silver's ingest_ts to BOTH initial_load_ts and last_update_ts
WHEN NOT MATCHED THEN
  INSERT (
    sale_id,
    product,
    category,
    quantity,
    price,
    sale_date,
    region,
    initial_load_ts,
    last_update_ts
  )
  VALUES (
    source.sale_id,
    source.product,
    source.category,
    source.quantity,
    source.price,
    source.sale_date,
    source.region,
    source.ingest_ts,
    source.ingest_ts
  );


In [0]:
%sql
select * from intellibi_catalog.intellibi_gold.MonthlySales 
